[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L16.ipynb)

# Lecture 16 — Train vs Eval, no_grad & Checkpointing

*Two toggles that change how your model behaves — and the safety net that keeps your training run alive.*

**PyTorch from Ground Up · Volume I · Chapters 28–29** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Say exactly what `model.train()` and `model.eval()` change — and what they don’t.
- Explain what `torch.no_grad()` saves, and why evaluation uses it together with `eval()`.
- Write the complete train + evaluate loop and the inference recipe.
- Track and plot train and validation loss, and log them to disk.
- Save and load a `state_dict` safely, and keep the best model.
- Save a full checkpoint and resume training exactly where it stopped.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · What train() and eval() actually do

Last lecture’s loop trained — but didn’t **evaluate** properly. Evaluation needs two switches. One from learning mode to inference mode, one to turn gradient tracking off.

<sub>Source: lecture slides 3–11</sub>

🧪 *Interactive lab in the app:* **Same input, twice** — try it there to see this idea move.

> ⚠️ **Watch out — Forgot eval() during validation**  
> Forget `model.eval()` during validation and Dropout is still active. Your validation accuracy will appear **lower than it really is**, because neurons are randomly silenced.
>
> This can mislead you into training longer or choosing a worse model.

## 02 · torch.no_grad(): what eval() doesn’t do

`eval()` changes layers — it does **not** stop the graph. `no_grad()` does. In eval mode every forward pass still records a computation graph and spends memory on it. You never call `.backward()` while evaluating, so that graph is pure waste.

<sub>Source: lecture slides 12–16</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
model = nn.Sequential(nn.Linear(10, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.5), nn.Linear(32, 3))
x = torch.randn(4, 10)
_X, _y = torch.randn(200, 10), torch.randint(0, 3, (200,))
train_loader = DataLoader(TensorDataset(_X[:160], _y[:160]), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(_X[160:], _y[160:]), batch_size=32)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
num_epochs = 3


**`no_grad.py`** — predict the output, then run.

In [ ]:
# Without no_grad: graph is built, memory used
model.eval()
out = model(x)
print(out.requires_grad)           # True — graph was built!
# With no_grad: no graph, less memory
with torch.no_grad():
    out = model(x)
print(out.requires_grad)           # False — no graph

<details><summary>▸ Expected output</summary>

```
True
False
```
</details>

## 03 · The complete train + evaluate loop

Three additions — **`train()`, `eval()`, `no_grad()`** — and the validation phase never steps. No `zero_grad`, no `backward`, no `step`: it measures, it doesn’t learn. Divide by `len(loader.dataset)`, the number of samples — not the number of batches.

<sub>Source: lecture slides 17–26</sub>

**`train_eval.py`** — predict the output, then run.

In [ ]:
for epoch in range(num_epochs):
    # ── Training phase ────────────────────
    model.train()                              # ❶
    train_loss = 0.0
    for bx, by in train_loader:
        pred = model(bx)
        loss = loss_fn(pred, by)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * bx.size(0)
    # ── Validation phase ──────────────────
    model.eval()                               # ❷
    val_loss = 0.0
    with torch.no_grad():                      # ❸
        for bx, by in val_loader:
            pred = model(bx)
            val_loss += loss_fn(pred, by).item() * bx.size(0)
    # ── Logging ───────────────────────────
    t = train_loss / len(train_loader.dataset)
    v = val_loss   / len(val_loader.dataset)
    print(f"Epoch {epoch:2d}  "
          f"Train: {t:.4f}  Val: {v:.4f}")

**`worked_example.py`** — predict the output, then run.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split
torch.manual_seed(0)
x = torch.randn(600, 10)
y = (x[:, 0] + x[:, 1] > 0).long()     # binary classification
ds = TensorDataset(x, y)
train_ds, val_ds = random_split(ds, [480, 120],
    generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, 32, shuffle=True)
val_loader   = DataLoader(val_ds,   32)

model = nn.Sequential(
    nn.Linear(10, 32), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(32, 2))
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(20):
    model.train()
    t_loss = 0.0
    for bx, by in train_loader:
        loss = loss_fn(model(bx), by)
        opt.zero_grad(); loss.backward(); opt.step()
        t_loss += loss.item() * bx.size(0)
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            v_loss += loss_fn(model(bx), by).item() * bx.size(0)
    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d}  "
              f"Train: {t_loss/480:.4f}  "
              f"Val: {v_loss/120:.4f}")

<details><summary>▸ Expected output</summary>

```
Epoch  0  Train: 0.6110  Val: 0.4610
Epoch  5  Train: 0.1028  Val: 0.0940
Epoch 10  Train: 0.0489  Val: 0.0609
Epoch 15  Train: 0.0390  Val: 0.0574
```
</details>

> ⚠️ **Watch out — Switch back to train()**  
> Forget to switch back to `model.train()` at the start of the next epoch, and the model stays in eval mode — Dropout is off during training. It trains without regularization and will overfit faster.
>
> Always toggle — `train()` before training, `eval()` before validation.

## 04 · Inference, and why the two calls are separate

Two switches, **two different jobs**: one changes layer behaviour, the other the graph. Rare cases want one without the other — computing gradients during evaluation, for example, to interpret a model.

<sub>Source: lecture slides 27–30</sub>

**`predict.py`** — predict the output, then run.

In [ ]:
model.eval()
with torch.no_grad():
    new_x = torch.randn(5, 10)
    logits = model(new_x)
    preds = logits.argmax(dim=1)
print(preds)

<details><summary>▸ Expected output</summary>

```
tensor([0, 1, 1, 0, 1])
(random input — your five predictions will differ)
```
</details>

## 05 · Tracking and logging

A long run can **die at any moment**. Three habits protect you. The machine crashes, the notebook disconnects, you lose track of the best epoch. **Tracking** shows what’s happening; **logging** survives a restart; **checkpointing** lets you resume.

<sub>Source: lecture slides 33–40</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
avg_train, avg_val = 0.50, 0.40   # stand-ins for the numbers your training and validation phases compute


**`history.py`** — predict the output, then run.

In [ ]:
history = {"train_loss": [], "val_loss": []}
for epoch in range(num_epochs):
    # ... training phase ...
    history["train_loss"].append(avg_train)
    # ... validation phase ...
    history["val_loss"].append(avg_val)

import matplotlib.pyplot as plt
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"],   label="val")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.show()

**`log_csv.py`** — predict the output, then run.

In [ ]:
import csv
with open("log.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss"])

for epoch in range(num_epochs):
    # ... training + validation ...
    with open("log.csv", "a", newline="") as f:
        csv.writer(f).writerow(
            [epoch, f"{avg_train:.6f}",
                    f"{avg_val:.6f}"])

## 06 · Saving and loading a state_dict

A model’s weights live in its **`state_dict`**: parameter names mapped to tensors. Save: one line. Load: two.

<sub>Source: lecture slides 41–47</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
def MyModel():   # stand-in: the same architecture as the worked example
    return nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 2))


**`save_load.py`** — predict the output, then run.

In [ ]:
# Save
torch.save(model.state_dict(), "model.pt")

# Load
model = MyModel()                       # same architecture
model.load_state_dict(
    torch.load("model.pt", weights_only=True))
model.eval()

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
)
for k, v in model.state_dict().items():
    print(f"{k:12s}  {v.shape}")

<details><summary>▸ Expected output</summary>

```
0.weight      torch.Size([32, 10])
0.bias        torch.Size([32])
2.weight      torch.Size([3, 32])
2.bias        torch.Size([3])
```
</details>

> ⚠️ **Watch out — Save the state_dict, not the model**  
> `torch.save(model, ...)` uses Python’s pickle, which ties the file to the exact class definition and file structure — rename a file or move a class, and loading breaks.
>
> A `state_dict` is portable — it is just a dictionary of tensors.

## 07 · Best model, full checkpoints, and when to save

Keep the **best** epoch, not the last. `best.pt` holds the weights from the lowest validation loss — even if the model overfitted afterwards. A simple form of early stopping.

<sub>Source: lecture slides 48–56</sub>

**`best_model.py`** — predict the output, then run.

In [ ]:
best_val = float("inf")
for epoch in range(num_epochs):
    # ... training ...
    # ... validation → avg_val ...
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), "best.pt")
        print(f"   ✓ Saved best (val={avg_val:.4f})")

🧪 *Interactive lab in the app:* **Track, save the best, crash, resume** — try it there to see this idea move.

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)


**`checkpoint.py`** — predict the output, then run.

In [ ]:
# Save a full checkpoint
checkpoint = {
    "epoch":      epoch,
    "model":      model.state_dict(),
    "optimizer":  optimizer.state_dict(),
    "best_val":   best_val,
    "history":    history,
}
torch.save(checkpoint, "checkpoint.pt")

# Resume from checkpoint
ckpt = torch.load("checkpoint.pt", weights_only=True)
model.load_state_dict(ckpt["model"])
optimizer.load_state_dict(ckpt["optimizer"])
start_epoch = ckpt["epoch"] + 1
best_val    = ckpt["best_val"]
history     = ckpt["history"]
for epoch in range(start_epoch, num_epochs):
    ...  # continue training exactly where you left off

## The rule — worth remembering

**train(), eval(), no_grad — six rules**

1. `model.train()` enables Dropout and live BatchNorm statistics. Use before every training phase.
2. `model.eval()` disables Dropout and uses stored BatchNorm statistics. Use before every evaluation phase.
3. `torch.no_grad()` disables gradient tracking. Use around any forward pass that won’t call `backward()`.
4. During evaluation, always use both `eval()` and `no_grad()` together.
5. Always switch back to `train()` at the start of each training epoch — forgetting is a silent bug.
6. For inference after training: `eval()` + `no_grad()` + forward pass + extract predictions.

**Tracking, logging, checkpointing — six rules**

1. Track train and val loss in a dict. Plot both curves — they are your diagnostic dashboard.
2. Log to a CSV (or TensorBoard) so numbers survive crashes.
3. Save `model.state_dict()`, not the model object. Use `weights_only=True` when loading.
4. Save the best model whenever validation loss improves.
5. For resumable training, save a full checkpoint: model + optimizer + epoch + best_val + history.
6. Always restore the optimizer state when resuming — without it, Adam’s buffers reset and training destabilizes.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — count the outputs

Create a model with `Dropout(0.5)`. Pass the same input 10 times in `train()` mode and count how many unique outputs you get. Switch to `eval()` and do the same.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

In train mode you get (almost certainly) 10 different outputs — a different set of neurons is zeroed each pass. In eval mode you get exactly 1.
</details>

### Exercise 02 · Try it — check against the rule

Run the worked example. After training, make predictions on 10 new random samples and compare them to the true rule `x[:,0] + x[:,1] > 0`. How many does the model get right?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

With validation loss near 0.06 the model has essentially learned the rule: expect 9 or 10 of 10, with mistakes only for points very close to the boundary where x₀ + x₁ ≈ 0.
</details>

### Exercise 03 · Ch 28 · Ex 3 & 5

Evaluate on the validation set without `model.eval()` — is the loss higher than with eval? Then deliberately forget to switch back to `model.train()` and train 20 more epochs. What happened?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Without `eval()` the validation loss is higher and changes from run to run, because Dropout is still silencing 30% of the neurons. Without switching back, the model trains with Dropout off — the training loss ends lower, but you have removed the regularization.
</details>

### Exercise 04 · Ch 28 · Ex 6

Add accuracy tracking to the validation phase; print accuracy alongside loss each epoch.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Inside the validation loop: `correct += (model(bx).argmax(dim=1) == by).sum().item()`; afterwards `acc = correct / len(val_loader.dataset)`.
</details>

### Exercise 05 · Ch 29 · Ex 2 & 3

Save the trained model’s `state_dict`, load it into a new instance, and verify the validation loss matches. Then change the hidden size and try to load the same file. What error do you get?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The reloaded model reproduces the validation loss exactly (remember `model.eval()`). With a different hidden size, `load_state_dict` raises `RuntimeError: … size mismatch for 0.weight …` — keys and shapes must match.
</details>

### Exercise 06 · Ch 29 · Ex 5 and best-model check

Add best-model saving; after training, load `best.pt` and confirm its validation loss equals the best you printed. Then write a full checkpoint, restart the kernel, load it and resume for 10 more epochs. Does the loss continue from where it stopped?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Yes to both: `best.pt` reproduces the best validation loss, and with model *and* optimizer state restored the curve continues smoothly from `start_epoch = ckpt["epoch"] + 1`.
</details>

### Interview-style

(a) “What is the difference between `model.eval()` and `torch.no_grad()`, and why do you need both during evaluation?” — four sentences; name the two layers.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`model.eval()` sets `self.training = False`, which changes how Dropout and BatchNorm behave: Dropout stops zeroing neurons and BatchNorm uses its stored running statistics. `torch.no_grad()` is unrelated to layer behaviour — it stops autograd from building a computation graph. During evaluation you need `eval()` so the outputs are deterministic and correct, and `no_grad()` so you don’t waste memory and time on a graph you will never backpropagate through. One changes behaviour; the other saves resources.
</details>

### Interview-style

(b) “Why save `model.state_dict()` instead of the model itself? And why must you also save the optimizer state when checkpointing for resume?” — four sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Saving the whole model pickles the object, tying the file to the exact class definition and file layout, so refactoring the code breaks loading. A `state_dict` is just a dictionary of named tensors — portable, and safe to load with `weights_only=True`. The optimizer has state too: momentum buffers and Adam’s running averages. If you resume without them they restart from zero, and the first updates are oversized, producing a spike in the loss.
</details>